# cabinsim on Google Colab

EB205 cabin noise model: engine and boundary-layer noise through the Boeing 787-9 sidewall,
with passive treatments, shunted piezo patches and an active trim panel.
This notebook installs the package, runs the model, plots the results and plays the cabin sound.

## 1. Install
Replace `<repository-url>` with the team's Git URL (skip this cell when running locally).

In [ ]:
!pip install -q "git+<repository-url>"

## 2. Run the model
`fast=True` takes about a minute; `fast=False` is the full-accuracy run.

In [ ]:
from cabinsim.params import load_params
from cabinsim.pipeline import run
from cabinsim.export import summary_rows

params = load_params()
results = run(params, fast=True)
for row in summary_rows(results):
    print(f"{row['label']:<22} head SIL {row['head_sil_db']:5.1f} dB   {row['head_dba']:5.1f} dB(A)   +{row['added_mass_kg_m2']:.2f} kg/m2")

## 3. Figures
The same figures as in the Streamlit app and `python -m cabinsim.run --plots`.

In [ ]:
from cabinsim import plots
plots.sil_figure(results)

In [ ]:
plots.cabin_spectrum_figure(results, "head")

In [ ]:
plots.transmission_loss_figure(results)

## 4. Listen
All cases share one gain, so the level differences are the model's differences.

In [ ]:
from IPython.display import Audio, display
from cabinsim.auralisation import synthesise, common_gain, wav_bytes

cases = synthesise(results, params.visuals)
gain = common_gain([c.signal_pa for c in cases], params.visuals["audio_headroom_db"])
for case in cases:
    print(case.label)
    display(Audio(wav_bytes(case.signal_pa, gain, int(params.visuals["audio_sample_rate"]))))

## 5. Change an input and compare
Example: a heavier septum in V1. Every value in `params` comes from the YAML files with its source tag.

In [ ]:
edited = params.copy()
edited.solutions = {key: edited.solutions[key] for key in ("V0_baseline", "V1_passive")}   # the copy's own dicts
edited.solutions["V1_passive"]["septum"]["surface_mass"] = 2.0
edited_results = run(edited, fast=True, log=lambda message: None)
for row in summary_rows(edited_results):
    print(f"{row['label']:<22} head SIL {row['head_sil_db']:5.1f} dB   {row['head_dba']:5.1f} dB(A)")

## 6. Assumptions still to verify

In [ ]:
for record in params.records:
    if record.source == "ASM" and not record.path.startswith("numerics."):
        print(f"{record.path:<45} {str(record.value):<20} {record.note[:70]}")